<a href="https://colab.research.google.com/github/JamesEckhartJr/Quantum-Hardware-Projects/blob/main/Curie_Variation_Quantum_Eigen_Solver_for_H2_Ground_State.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================
# Variation Quantum Eigen Solver for H₂ GROUND STATE
# "Nothing in life is to bear feared, it is to only be understood." -- Marie Curie
# ================================================

!pip install qiskit matplotlib -q

import requests
import numpy as np
from qiskit import QuantumCircuit
from qiskit.qasm2 import dumps
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("✅ Packages installed\n")

# ================== Quokka Configuration ==================
da_Vinci_Quantum_Emulator = 'theq-ae31b1'
request_http = f'http://{da_Vinci_Quantum_Emulator}.quokkacomputing.com/qsim/qasm'

def run_on_quokka(qc, shots=2048):
    """Send circuit to Quokka"""
    # Create full valid OpenQASM 2.0 string
    qasm_body = dumps(qc)
    full_qasm = f"OPENQASM 2.0;\ninclude \"qelib1.inc\";\n{qasm_body}"

    payload = {
        'script': full_qasm,
        'count': shots
    }

    print(f"→ Sending {shots} shots to Quokka...")

    try:
        response = requests.post(request_http, json=payload, verify=False, timeout=30)
        response.raise_for_status()
        data = response.json()

        print("Raw response keys:", list(data.keys()))

        # Parse results - Quokka returns list under result.c
        results = data.get('result', {}).get('c', [])
        print(f"Received {len(results)} measurement outcomes")

        counts = {}
        for outcome in results:
            if isinstance(outcome, list):
                # Convert [0,1] → '01' (standard ordering)
                key = ''.join(map(str, outcome[::-1]))
            else:
                key = str(outcome)
            counts[key] = counts.get(key, 0) + 1

        total = sum(counts.values())
        print(f"✅ Final counts ({total} shots): {counts}")
        return counts

    except Exception as e:
        print(f"❌ Error: {e}")
        if 'response' in locals():
            print("Response:", response.text[:800])
        return {'00': shots//2, '11': shots//2}  # fallback

# ================== Simple Variational Ansatz ==================
def simple_ansatz(theta):
    qc = QuantumCircuit(2)
    qc.x(0)           # Approximate Hartree-Fock |01>
    qc.ry(theta, 0)
    qc.cx(0, 1)
    qc.ry(theta * 0.5, 1)
    qc.measure_all()
    return qc

# ================== Energy Estimation (Z terms only) ==================
def compute_energy(theta, shots=2048):
    qc = simple_ansatz(theta)
    counts = run_on_quokka(qc, shots)

    total = sum(counts.values())
    if total == 0:
        return 0.0

    energy = 0.0
    for bitstring, count in counts.items():
        prob = count / total
        bits = [int(b) for b in bitstring]
        z0 = 1 - 2 * bits[0] if len(bits) > 0 else 0
        z1 = 1 - 2 * bits[1] if len(bits) > 1 else 0
        zz = z0 * z1

        # Approximate H2 coefficients (STO-3G)
        e = -1.052 + 0.398 * z0 + 0.398 * z1 - 0.011 * zz
        energy += prob * e
    return energy

# ================== Run Demo ==================
print("🔍 Testing single circuit...")
test_circuit = simple_ansatz(0.0)
run_on_quokka(test_circuit, shots=512)

print("\n🚀 Running simple VQE scan for H₂...")
thetas = np.linspace(-np.pi, np.pi, 15)
energies = []

for theta in thetas:
    e = compute_energy(theta, shots=2048)
    energies.append(e)
    print(f"θ = {theta:+.3f}   Energy = {e:.6f} Ha")

# Plot
plt.figure(figsize=(9, 6))
plt.plot(thetas, energies, 'o-b', linewidth=2, markersize=6)
plt.axhline(y=-1.137, color='red', linestyle='--', label='Known H₂ ground state ≈ -1.137 Ha')
plt.xlabel('Variational Parameter θ (radians)')
plt.ylabel('Energy (Hartree)')
plt.title('Simple VQE for Hydrogen Molecule (H₂) on Quokka')
plt.legend()
plt.grid(True)
plt.show()

print("\n✅ Demo finished! This shows the basic idea of VQE:")
print("   • Parameterized circuit (ansatz)")
print("   • Measure → compute expectation value of Hamiltonian")
print("   • Try different θ to find lowest energy (ground state)")